### Install new libraries

In [ ]:
# !pip install ddgs trafilatura
# !pip install --upgrade ddgs
# !pip install openai-agents

### Setup

In [ ]:

from agents import Agent, function_tool, handoff, Runner, trace
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
import base64
from concurrent.futures import ThreadPoolExecutor
# import chromadb
from ddgs import DDGS
from ddgs.exceptions import DDGSException, RatelimitException, TimeoutException
from dotenv import load_dotenv
# import gradio as gr
import json
# from litellm import completion
import os 
from openai import OpenAI

from IPython.display import display, Markdown
import pandas as pd
from pathlib import Path
from pprint import pprint
from pydantic import BaseModel
# import random
import re
import threading
# import requests

# import numpy as np
# import plotly.colors
# import plotly.graph_objects as go
import time
import trafilatura
from trafilatura.downloads import fetch_response
from trafilatura.settings import use_config
from urllib.parse import urlparse
import uuid

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API Key is missing")

openai_client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = "gpt-4.1-mini"
JUDGE_MODEL = 'gpt-5-mini'
IMAGE_MODEL = "gpt-image-2.5-flare"

ddgs = DDGS()

### Define the Tools

#### Helper Functions

In [ ]:
# Domains that refused a fetch during the current research run (cleared by run_research_agent)
BLOCKED_DOMAINS = set()

# URLs read successfully during the current research run (clear before each run)
READ_URLS = set()
REQUIRED_SOURCES = 4  # number of sources the researcher reads, then stops
MIN_SOURCE_CHARS = 2000  # shorter pages don't count as a source

# Statuses that mean only this page is missing, not that the site is blocking us
PAGE_ONLY_STATUSES = {404, 410}

# trafilatura defaults to a 30s timeout, and its retry backoff is half the timeout,
# so one unresponsive site can stall the agent for minutes. 10s keeps failures fast.
FETCH_CONFIG = use_config()
FETCH_CONFIG["DEFAULT"]["DOWNLOAD_TIMEOUT"] = "10"


def domain_of(url: str) -> str:
    """Return the URL's domain, lowercased and without a leading 'www.'."""
    return urlparse(url).netloc.lower().removeprefix("www.")


def source_progress() -> str:
    """Tell the agent how many sources it has read so far against the required minimum."""
    read, remaining = len(READ_URLS), REQUIRED_SOURCES - len(READ_URLS)
    if remaining > 0:
        return (f"[Progress: {read} of {REQUIRED_SOURCES} required sources read successfully. "
                f"Read at least {remaining} more before writing the report.]")
    return (f"[Progress: {read} of {REQUIRED_SOURCES} required sources read successfully. The requirement is met: "
            f"STOP searching and reading, and write the report now. "
            f"Cite only these sources: {', '.join(sorted(READ_URLS))}]")


def normalize_url(url: str) -> str:
    """Reduce a URL to domain + path + query so trivial differences (www, trailing slash) still match."""
    parts = urlparse(url.strip().rstrip(".,;"))
    query = f"?{parts.query}" if parts.query else ""
    return f"{domain_of(url)}{parts.path.rstrip('/')}{query}".lower()


def source_problem(brief: str):
    """Return feedback if the brief's Sources list cites unread or too few read sources, else None."""
    if "## Sources" not in brief:
        return ("Your reply is not a complete brief: it has no '## Sources' section. Reply with only "
                "the complete brief in the required format, with no preface.")
    sources = brief.split("## Sources", 1)[1]
    cited = {normalize_url(u) for u in re.findall(r"https?://[^\s)\]>]+", sources)}
    read = {normalize_url(u) for u in READ_URLS}
    unread, cited_read = sorted(cited - read), cited & read
    problems = []
    if unread:
        problems.append("It cites sources you did not read successfully with fetch_and_extract: "
                        + ", ".join(unread) + ". Read them or remove them.")
    if len(cited_read) < REQUIRED_SOURCES:
        problems.append(f"It cites only {len(cited_read)} sources you have read, but at least "
                        f"{REQUIRED_SOURCES} are required. Read more sources with fetch_and_extract.")
    if not problems:
        return None
    return ("Your brief does not meet the source requirement. " + " ".join(problems)
            + " Then reply with only the complete rewritten brief in the required format, with no preface.")

#### Web Search

In [ ]:
NUM_RESULTS = 10
MAX_PAGES = 3
MIN_SEARCH_INTERVAL = 2.5  # seconds between DuckDuckGo requests, across all agents and threads

_search_lock = threading.Lock()
_last_search_time = 0.0


def throttled_text(query: str, page: int):
    """Run one DuckDuckGo request at a time, spaced out, so parallel tool calls don't trigger rate limits."""
    global _last_search_time
    with _search_lock:
        wait = MIN_SEARCH_INTERVAL - (time.monotonic() - _last_search_time)
        if wait > 0:
            time.sleep(wait)
        try:
            return ddgs.text(query, max_results=NUM_RESULTS, page=page)
        finally:
            _last_search_time = time.monotonic()


@function_tool
def search_web(query: str):
    """Search the web with DuckDuckGo and return 10 results, retrying with exponential backoff."""
    retries, delay = 3, 2
    best = []
    for attempt in range(1, retries + 1):
        results, seen, error = [], set(), None
        # ddgs wraps all engine failures (including rate limits) in DDGSException;
        # TimeoutException is a subclass, so this catches both
        try:
            # A single page often returns fewer than max_results, so keep paging until we have enough
            for page in range(1, MAX_PAGES + 1):
                batch = throttled_text(query, page)
                if not batch:
                    break
                for result in batch:
                    href = result.get("href", "")
                    if href and href not in seen:
                        seen.add(href)
                        results.append(result)
                if len(results) >= NUM_RESULTS:
                    break
        except DDGSException as e:
            error = e

        # Keep the largest result set seen so far, even if a later page failed
        results = results[:NUM_RESULTS]
        if len(results) > len(best):
            best = results
        if len(best) == NUM_RESULTS:
            break
        if error is None:
            error = f"Only {len(best)} results found." if best else "No results found."

        if attempt < retries:
            print(f"  \N{WARNING SIGN} Search incomplete ({error}). Retrying in {delay}s...")
            time.sleep(delay)
            delay *= 2

    if best:
        # Flag results from sites that already blocked a fetch in this run
        for result in best:
            if domain_of(result.get("href", "")) in BLOCKED_DOMAINS:
                result["fetchable"] = False
        print(f"  \N{WHITE HEAVY CHECK MARK} Got {len(best)} results for {query}")
        return json.dumps(best, indent=2)

    print(f"  \N{CROSS MARK} Search failed after {retries} attempts: {error}")
    return f"Search for '{query}' failed ({error}). Try a different or broader query."

#### Scrape search results

In [ ]:
@function_tool
def fetch_and_extract(url: str):
    """Fetch the content of a URL and extract its text using Trafilatura.

    Domains that refuse a fetch are added to BLOCKED_DOMAINS and skipped for the rest of the run.
    """
    domain = domain_of(url)
    if url in READ_URLS:
        print(f"  \N{CLOCKWISE RIGHTWARDS AND LEFTWARDS OPEN CIRCLE ARROWS} Already read {url}.")
        return f"You already read {url}. Choose a different source.\n{source_progress()}"

    if domain in BLOCKED_DOMAINS:
        print(f"  \N{NO ENTRY SIGN} Skipped {domain}: blocked earlier in this run.")
        return (f"Skipped {url}: {domain} blocks fetching. Choose a source from a different site.\n"
                f"{source_progress()}")

    response = fetch_response(url, decode=True, config=FETCH_CONFIG)

    # No response (timeout, refused or reset connection) or an unexpected status such as
    # 403 or a 202 bot challenge means the site is refusing us, so its other pages will fail too
    if response is None or (response.status != 200 and response.status not in PAGE_ONLY_STATUSES):
        BLOCKED_DOMAINS.add(domain)
        reason = "no response" if response is None else f"HTTP {response.status}"
        print(f"  \N{CROSS MARK} {domain} blocked the fetch ({reason}).")
        return (f"Could not fetch {url} ({reason}). {domain} blocks fetching, so do not try other "
                f"pages on this site. Choose a source from a different site.\n{source_progress()}")

    text = trafilatura.extract(response.html) if response.status == 200 else None
    if not text:
        print(f"  \N{CROSS MARK} Failed to get text from URL.")
        return f"Could not fetch and extract text from {url}. Try a different source.\n{source_progress()}"

    if len(text) < MIN_SOURCE_CHARS:
        print(f"  \N{CROSS MARK} Too short ({len(text)} chars): {url[:60]}\n")
        return (f"{url} is too short ({len(text)} chars) to count as a source. "
                f"Choose a longer, more substantive article.\n{source_progress()}")

    READ_URLS.add(url)
    print(f"  \N{WHITE HEAVY CHECK MARK} fetch_and_extract: got {len(text)} chars from {url[:60]} "
          f"({len(READ_URLS)}/{REQUIRED_SOURCES} sources read)\n")
    return f"{source_progress()}\n\n{text}"

#### Generate Image

In [ ]:
@function_tool
def generate_image(prompt: str) -> str:
    """
    Generates an image and saves it to a local file.
    The prompt should be a detailed visual description.
    Returns the file path of the saved image.
    """
    print(f"  🎨 Generate image: {prompt[:60]}...")
    try:
        response = openai_client.images.generate(
            model=IMAGE_MODEL,
            prompt=prompt,
            size="1536x1024",
            quality="high",
            output_format="jpeg",
            n=1
        )
    except Exception as e:
        print(f"   ❌ generate image failed: {e}")
        return f"Image generation failed: {e}"
    # GPT image models return base64 data, not a URL
    Path("images").mkdir(exist_ok=True)
    image_path = f"images/hero_{uuid.uuid4().hex[:8]}.jpg"
    Path(image_path).write_bytes(base64.b64decode(response.data[0].b64_json))
    print(f"   ✅ generate image: Done ({image_path})")
    return f"Image generated successfully: {image_path}"

### The Agents

This tells the LLM who it is and how to behave. The key things:
- What its job is
- What tools it has
- What process to follow
- What output format to produce

#### Research Agent

In [ ]:
RESEARCH_AGENT_PROMPT = f"""
You are a research agent. Your job is to research a topic the user gives you by searching the web,
reading the most relevant sources, and writing a well-organized, factual research report based on
what you find.

## Tools
You have two tools:
- search_web(query): Searches the web with DuckDuckGo and returns up to 10 results, each with a
  title, URL (href) and short snippet (body). Use it to discover sources.
- fetch_and_extract(url): Fetches a web page and returns its main text. Use it to read a source in
  full. Some pages cannot be fetched, or are too short (under {MIN_SOURCE_CHARS} characters) to count
  as a source; if that happens, move on to a different source. Every result
  starts with a [Progress: ...] line showing how many sources you have read successfully so far.

*** IMPORTANT:
Always state your reasoning in plain text BEFORE you call any tool, so each step is explained.
- Before each search_web call: say which question or gap you are addressing and why this query
  is the right one.
- After each search, you MUST first explain your reasoning:
  - Which URLs look most relevant and why
  - Which ones you will fetch and why
  - Which ones you are skipping and why
  Only AFTER writing out your reasoning should you call fetch_and_extract.
- After each fetch (especially a failed one): say what you learned or why it failed, what is
  still missing, and what you will do next and why.
- Whenever you decide NOT to call a tool (for example, skipping a result, not searching again, or
  moving on to writing), say so explicitly and explain why.
- Before you write the report: say why you believe you have enough sources.
***

## Process
1. Plan: Break the topic into 2-4 key questions or angles that need answering.
2. Search: Run search_web for each angle. Use specific, focused queries.
3. Reflect on the results: Which sources look most relevant and credible, and why? Prefer
   authoritative sources (peer-reviewed research, government and academic sites, established news
   outlets).
4. Read: Use fetch_and_extract on the best URLs until you have successfully read {REQUIRED_SOURCES}
   different sources of sufficient length (at least {MIN_SOURCE_CHARS} characters each). If a fetch fails
   or the page is too short, replace it with another credible result. Search snippets
   are not enough; base your findings on the full text of the sources.
   As soon as you have read {REQUIRED_SOURCES} sufficient sources, STOP searching and reading.
   Do not look for more sources, even if you think more would help.
5. Reflect on what you have gathered: Do sources agree or conflict? Note any gaps in the key
   questions, but only search again if you have fewer than {REQUIRED_SOURCES} sufficient sources.
6. Fill the gaps: If you have fewer than {REQUIRED_SOURCES} sufficient sources, search again with a
   different query and repeat steps 3-5.
7. Synthesize: Once you have successfully read {REQUIRED_SOURCES} sufficient sources, synthesize
   them into a research report right away. Mention any gaps as limitations in the Conclusion.

## Rules
- Only state facts that come from the sources you read. Do not invent facts, statistics, quotes
  or URLs.
- If you could not find reliable information on part of the topic, say so.
- Cite sources inline with numbered references like [1], [2] that match the Sources list.
- The report MUST cite at least {REQUIRED_SOURCES} distinct sources. Do not cite the same source twice under
  different numbers, and do not cite sources you did not read.
- Do not call the same tool with the same arguments twice.

## Output Format

The report MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Write the report in Markdown using this structure:

# <Report title>

## Summary
A 2-3 sentence overview of the main findings.

## Key Findings
3-6 subsections, each with a ### heading and a short paragraph or bullet points with inline
citations.

## Conclusion
A brief wrap-up, including open questions or limitations of the research.

## Sources
A numbered list of at least {REQUIRED_SOURCES} distinct sources you used, in the format: [1] Title - URL

Until you are ready to deliver the report, just keep working: search, fetch, think, reflect.
Do not rush between tool calls, but once you have {REQUIRED_SOURCES} sufficient sources, write the report.
Not every response needs a tool call; sometimes just thinking through what you have is the right move.
"""

research_agent = Agent(
    name="research_agent",
    instructions=RESEARCH_AGENT_PROMPT,
    model = MODEL,
    tools = [search_web, fetch_and_extract],
)

research_agent_as_tool = research_agent.as_tool(
        tool_name="research_agent",
        tool_description=(
            "Researches a topic and returns a research brief in Markdown. "
            "Give it a topic or question as input. It plans the key angles, searches the web, "
            f"reads the full text of {REQUIRED_SOURCES} distinct sources, and checks for gaps before writing. "
            "The brief contains a title, Summary, Key Findings (with key facts, statistics and "
            "data points), Conclusion (including limitations), and a numbered Sources list with "
            "URLs, with inline [n] citations. Only facts found in the sources it read are included. "
            "Each call is independent and quality can vary, so results can differ between calls."
        ),
        max_turns=25
    )

#### Image Generator Agent

In [ ]:
IMAGE_AGENT_PROMPT = f"""
You are an image prompt specialist. Given a topic and content summary,
craft a detailed {IMAGE_MODEL} prompt for a hero image.

Rules for your {IMAGE_MODEL} prompt:
- Describe a natural, photographic-style image (not illustrated, not cartoon)
- No text, logos, or words in the image
- No recognizable people
- No icon dumps or collages
- Focus on a single compelling visual that captures the topic
- Be specific about lighting, composition, and mood
- Keep the prompt under 200 words

Call generate_image exactly ONCE with your prompt. One image only.
Return the image file path exactly as given by generate_image.
"""

image_generator_agent = Agent(
    name="image_generator_agent",
    instructions=IMAGE_AGENT_PROMPT,
    model = MODEL,
    tools = [generate_image],
)

image_agent_as_tool = image_generator_agent.as_tool(
    tool_name = "image_generator_agent",
    tool_description = "Generates a hero image for an article based on a topic and content summary."
                  "Supply the topic and content summary for the article."
)

#### Writer Agent A: The Journalist

In [ ]:
JOURNALIST_WRITER_PROMPT = RECOMMENDED_PROMPT_PREFIX + """
You are an investigative journalist that writes provocative articles. 
You will receive a conversation history that includes research on a topic. 

Your style: 
- Lead with the most surprising or controversial finding — your opening should grab the reader 
- Challenge assumptions and ask hard questions throughout 
- Take a clear stance — you have an opinion and you're not afraid to share it 
- Quote sources and reference specific data points 
- Write in a conversational, punchy tone — short paragraphs, varied sentence length 
- Structure like a news feature: hook, context, evidence, tension, conclusion 
- Aim for 800-1200 words 
- List the sources you referenced at the end of the article

Do NOT use generic section headers like "Introduction", "Conclusion", or "Overview". Use creative, specific headers that grab attention.
Do NOT overuse headers. Only use one title and BETWEEN 2 and 3 haders in total for the entire article.
Do NOT use bullet points or numbered lists.
Do NOT ask for feedback, offer revisions, or include any commentary after the article. 

If an image path was provided in the handoff, place the image at the very top, before the article, in markdown format:
![hero image](<path provided in the handoff>)
If no image was provided, do NOT create, invent, or reference an image. Only use an image if one is provided.
"""

journalist_agent = Agent(
    name="journalist_agent",
    instructions=JOURNALIST_WRITER_PROMPT,
    model = MODEL,
    )

#### Writer Agent B: Storyteller

In [ ]:
STORYTELLER_WRITER_PROMPT = RECOMMENDED_PROMPT_PREFIX + """
You are a storyteller who writes magazine longform articles. 
You will receive a conversation history that includes research on a topic. 

Your style: 
- Write as a narrative with characters, scenes, and dialogue — it should read like a magazine longform piece 
- Open in the middle of a specific scene: a place, a moment, a person doing something concrete 
- Build the article around one or two composite characters whose experience illustrates the research 
- Weave the research findings, data points, and quotes into the story naturally, through what characters see, say, and decide 
- Use vivid sensory detail and dialogue to bring scenes to life, and vary pacing between scene and reflection 
- Make clear to the reader when a character or scene is illustrative rather than a documented real person or event 
- Never invent statistics, studies, or quotes from real sources — every factual claim must come from the research 
- Structure like a longform feature: scene, widening context, complication, resolution, reflection 
- Aim for 800-1200 words 
- List the sources you referenced at the end of the article

Do NOT use generic section headers like "Introduction", "Conclusion", or "Overview". Use evocative, specific headers.
Do NOT overuse headers. Only use one title and NO MORE THAN 2 to 3 headers in total for the entire article.
Do NOT use bullet points or numbered lists in the article body.
Do NOT ask for feedback, offer revisions, or include any commentary after the article. 

If an image path was provided in the handoff, place the image at the very top, before the article, in markdown format:
![hero image](<path provided in the handoff>)
If no image was provided, do NOT create, invent, or reference an image. Only use an image if one is provided.
"""

storyteller_agent = Agent(
    name="storyteller_agent",
    instructions=STORYTELLER_WRITER_PROMPT,
    model = MODEL,
    )

In [ ]:
# with trace("Storyteller test", group_id="Learning AI Engineering"):
#     result = await Runner.run(
#         storyteller_agent,
#         input="Write an article about the following topic: Midterm losses for the sitting presidents party.",
#         max_turns=30,
#     )


In [ ]:
# display(Markdown(result.final_output))

#### Orchestrator Agent

In [ ]:
ORCHESTRATOR_AGENT_PROMPT = RECOMMENDED_PROMPT_PREFIX + """
You are the orchestrator of a multi-agent article writing system.
Your job is to coordinate tools and other agents to produce a high-quality article. 
Use the tools available to you and/or delegate tasks to the appropriate agents.
Never do the work yourself. Always use tools or agents. 
Your tools and agents are specialists and should be doing the work, you are the manager.

You use the research_agent tool twice (and ONLY twice) with slightly varying inputs to get 2 research briefs.
You pick the best research brief out of the two and ALWAYS deliver the selected brief as output. 
Do not combine the two briefs, just pick the best one.
Do not do the research yourself or add anything, you MUST use the research_agent tool to get the briefs.

After picking the best brief, your NEXT step MUST be to call the image_generator_agent tool. This is mandatory.
Do not write your final answer until image_generator_agent has returned a result.
Pass it the topic and a short content summary taken from the selected brief.

The image_generator_agent returns a file path. Only use a path that image_generator_agent actually returned.
Never invent, guess, or reuse a file path. If it returns an error, say so instead of making up a path.

Then hand off to ONE writer, based on the topic, along with the selected research brief and the image path returned by image_generator_agent:
- journalist_agent: provocative, opinionated, news-feature style. Best for contested, political, or controversial topics, or where the research challenges conventional wisdom.
- storyteller_agent: narrative longform with characters and scenes. Best for human-centered, cultural, or emotional topics, or where the research is best understood through people's experiences.
Think about which style would serve the topic and the findings in the brief best, then choose.
"""

orchestrator_agent = Agent(
    name="orchestrator_agent w/ handoff and styles",
    instructions=ORCHESTRATOR_AGENT_PROMPT,
    model=MODEL,
    tools=[research_agent_as_tool, image_agent_as_tool],
)

#### Testing Orchestrator
Removes tool calls to expedite testing.

In [ ]:
# TEMPORARY: modified orchestrator without tools, for QUICK TESTING of the HANDOFF
ORCHESTRATOR_AGENT_PROMPT = RECOMMENDED_PROMPT_PREFIX + """
You are the orchestrator of a multi-agent article writing system.
Your job is to coordinate tools and other agents to produce a high-quality article. 
IGNORE the tools available to you.

Then, decide which writer is the best fit for this topic:
- journalist_agent: provocative, opinionated, news-feature style. Best for contested, political, or controversial topics, or where the research challenges conventional wisdom.
- storyteller_agent: narrative longform with characters and scenes. Best for human-centered, cultural, or emotional topics, or where the research is best understood through people's experiences.

Then hand off to your chosen writer.
Do not write the article yourself, you MUST hand off to a writer.
"""

orchestrator_agent = Agent(
    name="orchestrator_agent w/ handoff and styles (testing)",
    instructions=ORCHESTRATOR_AGENT_PROMPT,
    model=MODEL,
)

#### Update the Orchestrator Agent

In [ ]:
class WriterSelectionInfo(BaseModel):
    which_writer: str
    reason: str

async def on_handoff(_ctx, decision: WriterSelectionInfo):
    print(f"  ✍️ Writer selected: {decision.which_writer}, Reason: {decision.reason}")

orchestrator_agent.handoffs = [
    handoff(agent=journalist_agent, on_handoff=on_handoff, input_type=WriterSelectionInfo), 
    handoff(agent=storyteller_agent, on_handoff=on_handoff, input_type=WriterSelectionInfo)
]

### Let's Run It!

In [ ]:
# Expected writer is a guess; the orchestrator decides at run time
TEST_TOPICS = {
    "Why the sitting president's party almost always loses seats in the midterms": "journalist_agent",
    "Is the electric vehicle boom a climate solution or a mining problem?": "journalist_agent",
    "The hidden costs of college sports and who really profits": "journalist_agent",
    "What life is like for family caregivers of people with dementia": "storyteller_agent",
    "The people who keep small-town newspapers alive": "storyteller_agent",
    "How immigrant families have shaped American food culture": "storyteller_agent",
}

# pick on topic from the dictionary with the user picking the index
test_topic = list(TEST_TOPICS.keys())[5]
print(test_topic)
expected_writer = TEST_TOPICS[test_topic]
print(expected_writer)

In [ ]:
with trace("Article Writer w/ styles", 
           group_id="AIEP",
           metadata={"experimentID": "001"}):
    result = await Runner.run(
        orchestrator_agent,
        input=test_topic,
        max_turns = 30
    )

In [ ]:
print(f"Agent result: {result.last_agent.name}")
print(f"---")
display(Markdown(result.final_output))

### Evals

Evals Best practices:

1. Code-based where possible
2. Just Output vs Traces
3. Involve the product person
4. One judge per failure mode
5. Binary score is better
6. Run evals across diverse inputs

In [ ]:
JUDGE_PROMPT = """
# Judge Prompt: Insufficient Source Breadth (TRUE/FALSE)

You are scoring a research agent **transcript/trace** to determine if the final **research brief** failed to reference at least 3 different sources.
Return **only** `TRUE` or `FALSE`.

## Definitions

**Research brief:** The final deliverable the agent hands back to the user. Evaluate only the brief itself, not intermediate tool calls, search results, scratch notes, or reasoning steps in the trace.

**Source:** Any identifiable external origin of information that the brief references, in any form. Formal citation is NOT required. All of the following count:
- A URL or hyperlink
- A named publication, organization, report, dataset, paper, or website (e.g., "according to the Federal Reserve's 2025 Beige Book," "Gartner estimates," "a McKinsey survey found")
- A footnote, endnote, or "Sources" list entry
- An inline mention that ties a specific claim to a specific named origin

**Distinct source:** A separate document or publication. Count each one once, no matter how many times it is referenced.
- The same article cited three times = 1 source
- Two different articles from the same outlet or domain = 2 sources
- The same report referenced by full title in one place and by short name in another = 1 source

**Does NOT count as a source:**
- Vague attributions with no identifiable origin ("studies show," "experts say," "industry data suggests," "it is widely reported")
- The user's own prompt, instructions, or files the user supplied as task input
- The agent's own analysis, reasoning, or general knowledge
- Sources that appear only in the trace (searched, fetched, or read) but are never referenced in the brief

## Insufficient breadth (label TRUE): Any of the below occurred

1. **Fewer than 3 distinct sources:** After deduplication, the brief references 2 or fewer distinct sources.
2. **Padding through repetition:** The brief appears to reference 3 or more sources, but after applying the deduplication rules above the count falls below 3.
3. **Vague attribution only:** The brief relies on unattributed claims ("research indicates...") such that fewer than 3 identifiable sources remain.
4. **Research done but not used:** The trace shows the agent consulted 3 or more sources, but the brief references fewer than 3 of them.
5. **No brief delivered:** The agent did not produce a research brief, or the deliverable is empty or truncated before any substantive content.

## Sufficient breadth (label FALSE)

The brief references **3 or more distinct, identifiable sources**, counted under the rules above. Do not penalize for citation format, citation style, missing dates, missing page numbers, or source quality. Do not attempt to verify whether a source exists or supports the claim it is attached to. This judge measures breadth only.

## Output Format

Return exactly one token: `TRUE` or `FALSE`. No explanations.
"""

In [ ]:
TOPICS = [
    "The current state of solid-state battery commercialization for EVs",
    "Effectiveness of GLP-1 drugs for weight loss beyond the first two years",
    "How tariffs enacted since 2025 have affected US consumer goods prices",
    "The causes of the Bronze Age collapse around 1200 BCE",
    "Remote work's long-term impact on commercial real estate valuations",
    "CRISPR-based therapies approved for sickle cell disease and their access challenges",
    "The economics of NIL deals in college athletics",
    "Microplastics in drinking water: health risks and filtration options",
    "Small modular nuclear reactors: cost, timelines, and regulatory hurdles",
    "Historical performance of the Dogs of the Dow strategy",
]

In [ ]:
import contextlib
import io
import sys

notebook_stdout = sys.stdout

def judge(brief: str) -> str:
    resp = client.responses.create(
        model=JUDGE_MODEL,
        instructions=JUDGE_PROMPT,
        input=f"<research_brief>\n{brief}\n</research_brief>",
    )
    label = resp.output_text.strip().strip("`").upper()
    return label if label in {"TRUE", "FALSE"} else f"INVALID: {resp.output_text!r}"

def evaluate(topic: str) -> dict:
    print(f"[{TOPICS.index(topic)+1}/{len(TOPICS)}] Researching: {topic[:60]}...", file=notebook_stdout, flush=True)
    try:
        brief = run_research_agent(topic, max_iterations=10)
        return {"topic": topic, "failure": judge(brief), "brief": brief}
    except Exception as e:
        return {"topic": topic, "failure": "ERROR", "brief": repr(e)}

# redirect_stdout swaps sys.stdout process-wide, so wrap the whole pool once instead of each thread
with contextlib.redirect_stdout(io.StringIO()), ThreadPoolExecutor(max_workers=5) as pool:
    df = pd.DataFrame(pool.map(evaluate, TOPICS))

print(df["failure"].value_counts().to_string(), "\n")
print(f"Failure rate (scored runs): {(df.failure == 'TRUE').sum() / df.failure.isin(['TRUE', 'FALSE']).sum():.0%}")
df[["topic", "failure"]]